# 实验四：基于大语言模型的人类编写代码的代码审查

## 阶段一：数据准备

从 Lab1 的原始数据中，挑选有 Review Comment 的 PR，随机抽取 200 个，保存到 `lab4/data/` 下。

### 导入依赖 & 执行数据准备

In [1]:
import sys
import os

sys.path.insert(0, os.getcwd())
from data_preparer import run_data_preparation

data, summary = run_data_preparation()

随机种子: 42
挑选数量: 100
Lab1 数据目录: f:\学习资料\智能软件工程实践\lab1\data\raw
Lab4 输出目录: f:\学习资料\智能软件工程实践\lab4\data

步骤一：加载 Lab1 全部原始数据
  加载 kubernetes_kubernetes: 300 PRs
  加载 pytorch_pytorch: 300 PRs
  加载 tensorflow_tensorflow: 300 PRs
  加载 microsoft_vscode: 300 PRs
  加载 langchain-ai_langchain: 300 PRs

  共加载 1500 个 PR

步骤二：筛选有 Review Comment 的 PR
有 Review Comment 的 PR: 801 / 1500
占比: 53.4%

步骤三：随机抽取
随机抽取 100 个 PR

步骤四：提取 Lab4 所需字段
提取完成，共 100 条数据

步骤五：保存数据
数据已保存到: f:\学习资料\智能软件工程实践\lab4\data\selected_prs.json
文件大小: 3.15 MB

步骤六：计算统计摘要
统计摘要已保存到: f:\学习资料\智能软件工程实践\lab4\data\summary.json


### 数据集概览

In [2]:
print("=" * 60)
print("数据集概览")
print("=" * 60)
print(f"总 PR 数: {summary['total_prs']}")
print()

print("按仓库分布:")
for repo, count in sorted(summary['repo_distribution'].items(), key=lambda x: -x[1]):
    print(f"  {repo}: {count} PRs")
print()

print(f"合并状态: {summary['merge_distribution']['merged']} merged / {summary['merge_distribution']['not_merged']} not merged")
print(f"合并率: {summary['merge_rate'] * 100:.1f}%")
print()
print(f"总评论数: {summary['total_comments']}")
print(f"平均评论数/PR: {summary['avg_comments_per_pr']:.1f}")
print(f"不同 Reviewer 数: {summary['unique_reviewers']}")
print()

ds = summary['diff_length_stats']
print(f"Diff 长度: min={ds['min']}, max={ds['max']}, mean={ds['mean']:.0f}")
bs = summary['body_length_stats']
print(f"PR Body 长度: min={bs['min']}, max={bs['max']}, mean={bs['mean']:.0f}")

数据集概览
总 PR 数: 100

按仓库分布:
  kubernetes_kubernetes: 35 PRs
  microsoft_vscode: 33 PRs
  pytorch_pytorch: 20 PRs
  langchain-ai_langchain: 12 PRs

合并状态: 44 merged / 56 not merged
合并率: 44.0%

总评论数: 564
平均评论数/PR: 5.6
不同 Reviewer 数: 64

Diff 长度: min=280, max=152883, mean=18120
PR Body 长度: min=0, max=7906, mean=1688


### 数据样例展示（前 3 条）

In [3]:
from data_preparer import print_sample

print_samples = min(3, len(data))
for i in range(print_samples):
    print_sample(data, i)

print("✅ 阶段一：数据准备完成！")

--- 样例 1 ---
  PR ID: 322940
  仓库: microsoft_vscode
  标题: Add terminal test to chat scenario tests
  是否合并: ❌ 否
  修改文件数: 5
  +299 / -64
  Reviews 数: 1
  行内评论数: 3
  Issue 评论数: 0
  PR 描述长度: 0 字符
  Diff 长度: 20484 字符
  Commit 数量: 1

--- 样例 2 ---
  PR ID: 139891
  仓库: kubernetes_kubernetes
  标题: [kubelet] DRA PrepareDynamicResources failures should be reported in SyncResult
  是否合并: ❌ 否
  修改文件数: 3
  +11 / -4
  Reviews 数: 2
  行内评论数: 2
  Issue 评论数: 13
  PR 描述长度: 1088 字符
  Diff 长度: 3769 字符
  Commit 数量: 2

--- 样例 3 ---
  PR ID: 140015
  仓库: kubernetes_kubernetes
  标题: Replace '#success' and '#failure' with 'successThreshold' and 'failureThreshold' for probes
  是否合并: ✅ 是
  修改文件数: 1
  +1 / -1
  Reviews 数: 0
  行内评论数: 0
  Issue 评论数: 5
  PR 描述长度: 3808 字符
  Diff 长度: 869 字符
  Commit 数量: 1

✅ 阶段一：数据准备完成！


---
## 阶段二：构建代码上下文

针对每条 PR，构建 4 种不同信息量的文本输入，测试信息量对 LLM 结果的影响：

| 类型 | 内容 | 信息量 |
|------|------|--------|
| `diff_only` | 仅 Diff | 最少 |
| `diff_pr_desc` | Diff + PR 标题 + PR 描述 | 中等 |
| `diff_commit` | Diff + Commit Message | 中等 |
| `diff_extra` | Diff + 文件名 + 修改函数 + 历史评论 | 最多 |

### 执行上下文构建

In [4]:
from context_builder import run_context_building

contexts, context_stats = run_context_building(data)

阶段二：构建代码上下文
4 种上下文类型: ['diff_only', 'diff_pr_desc', 'diff_commit', 'diff_extra']

  已处理 50/100 条...
  已处理 100/100 条...
上下文构建完成，共 100 条
上下文数据已保存到: f:\学习资料\智能软件工程实践\lab4\data\contexts.json
文件大小: 8.08 MB


### 上下文长度统计

In [5]:
from context_builder import print_context_stats

print_context_stats(context_stats)

各上下文类型长度统计

diff_only:
  样本数: 100
  最小长度: 280 字符
  最大长度: 152,883 字符
  平均长度: 18,120 字符
  总字符数: 1,812,039 字符

diff_pr_desc:
  样本数: 100
  最小长度: 779 字符
  最大长度: 153,404 字符
  平均长度: 19,939 字符
  总字符数: 1,993,879 字符

diff_commit:
  样本数: 100
  最小长度: 562 字符
  最大长度: 153,004 字符
  平均长度: 18,637 字符
  总字符数: 1,863,717 字符

diff_extra:
  样本数: 100
  最小长度: 834 字符
  最大长度: 161,995 字符
  平均长度: 23,364 字符
  总字符数: 2,336,431 字符


### 上下文样例展示（前 2 条 PR，各 4 种上下文）

In [6]:
from context_builder import print_context_sample

print_samples = min(2, len(contexts))
for i in range(print_samples):
    print_context_sample(contexts, i)

print("✅ 阶段二：上下文构建完成！")

--- PR #322940 (microsoft_vscode) ---

  [diff_only] 长度: 20,484 字符
  预览: --- a/build/azure-pipelines/linux/steps/product-build-linux-test.yml\n+++ b/build/azure-pipelines/linux/steps/product-build-linux-test.yml\n@@ -131,6 +131,18 @@ steps:\n       timeoutInMinutes: 20\n       displayName: 🧪 Run smoke tests (Electron)\n \n+  - ${{ if eq(parameters.VSCODE_RUN_ELECTRON_TESTS, tr...

  [diff_pr_desc] 长度: 20,617 字符
  预览: # Pull Request\n\n## Title\nAdd terminal test to chat scenario tests\n\n## Description\n(No description provided)\n\n## Code Changes (Diff)\n\n--- a/build/azure-pipelines/linux/steps/product-build-linux-test.yml\n+++ b/build/azure-pipelines/linux/steps/product-build-linux-test.yml\n@@ -131,6 +131,18 @@ steps:\n ...

  [diff_commit] 长度: 20,569 字符
  预览: # Commit Messages\n\nAdd terminal test to chat scenario tests\n\n## Code Changes (Diff)\n\n--- a/build/azure-pipelines/linux/steps/product-build-linux-test.yml\n+++ b/build/azure-pipelines/linux/steps/product-build-linux-test

---
## 阶段三：设计 Prompt

针对每种上下文，设计 4 种 Prompt 指令模板，共 4 × 4 = 16 组实验：

| Prompt 类型 | 策略 | 特点 |
|-------------|------|------|
| zero_shot | 直接问，不给示例 | 最简单 |
| few_shot | 先给 2 个示例，再问 | 提供参考 |
| cot (Chain-of-Thought) | 要求逐步分析 | 推理过程 |
| role_based | 赋予资深 Committer 角色 | 身份设定 |

### Prompt 样例展示

In [7]:
from prompt_builder import print_prompt_examples

print_prompt_examples(contexts)

Prompt Examples (Merge Prediction, diff_pr_desc context)

--- zero_shot ---
  Length: 21,001 chars
  Preview: Determine whether the following Pull Request code change is likely to be merged.\n\nIMPORTANT: Your ENTIRE response must be a single JSON object and nothing else. Do NOT wrap the JSON in ```json``` or any other markdown. Do NOT write any analysis, summary, introduction, or explanation. Just the JSON on its own line:\n{"decision": "Yes" or "No", "reason": "brief reason"}\n\nCode Change:\n# Pull Request\n\n## Title\nAdd terminal test to chat scenario tests\n\n## Description\n(No description provided)\n\n## Code...

--- few_shot ---
  Length: 21,652 chars
  Preview: Determine whether the following Pull Request code change is likely to be merged.\n\nHere are two examples to guide your judgment and output format:\n\nExample 1:\nCode Change:\n@@ -10,6 +10,6 @@ def calculate(a, b):\n-    return a + b\n+    if b == 0:\n+        return 0\n+    return a + b\nOutput:\n{"decision": "Yes", 

---
## 阶段四：模型推理

调用 DeepSeek API，完成 Merge Prediction 和 Review Comment Generation 两个任务。

**实验矩阵**：2 任务 × 4 上下文 × 4 Prompt = 32 组实验

**注意**：此步骤耗时较长，约 200 PR × 4 × 4 × 2 = 6400 次 API 调用。

### 执行全部实验

In [ ]:
import importlib
import config
importlib.reload(config)

from llm_inference import run_all_experiments, save_results

results = run_all_experiments(contexts)
save_results(results)

已加载已有结果: 2800 条记录

组 1/8: merge_prediction + zero_shot
  文件: f:\学习资料\智能软件工程实践\lab4\results\merge_prediction_zero_shot.json


[merge_prediction][zero_shot]: 100%|███████████| 400/400 [00:00<00:00, 476.13req/s, new=0, skip=400]


  跳过 400 条已有记录
  加载已有 400 条，新增 0 条
  本组完成: 400 条记录

  预测分布: Yes=288, No=63, Unknown=49
  Accuracy : 0.5300
  Precision: 0.4792
  Recall   : 0.7841
  F1-score : 0.5948
  ROC-AUC  : 0.5691

组 2/8: merge_prediction + few_shot
  文件: f:\学习资料\智能软件工程实践\lab4\results\merge_prediction_few_shot.json


[merge_prediction][few_shot]: 100%|████████████| 400/400 [00:00<00:00, 451.64req/s, new=0, skip=400]


  跳过 400 条已有记录
  加载已有 400 条，新增 0 条
  本组完成: 400 条记录

  预测分布: Yes=312, No=88, Unknown=0
  Accuracy : 0.5100
  Precision: 0.4679
  Recall   : 0.8295
  F1-score : 0.5984
  ROC-AUC  : 0.5442

组 3/8: merge_prediction + cot
  文件: f:\学习资料\智能软件工程实践\lab4\results\merge_prediction_cot.json


[merge_prediction][cot]: 100%|█████████████████| 400/400 [00:00<00:00, 466.33req/s, new=0, skip=400]


  跳过 400 条已有记录
  加载已有 400 条，新增 0 条
  本组完成: 400 条记录

  预测分布: Yes=124, No=253, Unknown=23
  Accuracy : 0.5100
  Precision: 0.4194
  Recall   : 0.2955
  F1-score : 0.3467
  ROC-AUC  : 0.4928

组 4/8: merge_prediction + role_based
  文件: f:\学习资料\智能软件工程实践\lab4\results\merge_prediction_role_based.json


[merge_prediction][role_based]:  33%|▎| 133/400 [20:19<34:14,  7.69s/req, new=133, lat=6.9s, skip=0]

### 实验结果统计

In [ ]:
from llm_inference import print_result_summary

print_result_summary(results)
print("\n✅ 阶段三+阶段四完成！")